# dee.cpp Ornith Milestone 2 proof
This notebook is intentionally thin. It checks the assigned machine, checks out one immutable repository revision, builds/tests it, and runs the repository-owned validation command.

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import psutil, torch
print(json.dumps({
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda_runtime': torch.version.cuda,
    'cpu_count': os.cpu_count(), 'ram_bytes': psutil.virtual_memory().total,
    'working_disk': shutil.disk_usage('/kaggle/working')._asdict(),
    'gpu_count': torch.cuda.device_count(),
    'gpus': [{'index': i, 'name': torch.cuda.get_device_name(i),
              'memory': torch.cuda.get_device_properties(i).total_memory}
             for i in range(torch.cuda.device_count())],
}, indent=2), flush=True)
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, f'dual-T4 assignment required, got {torch.cuda.device_count()} GPUs'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), [torch.cuda.get_device_name(i) for i in range(2)]

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                'transformers==5.14.1', 'safetensors==0.8.0',
                'pybind11==3.0.1', 'psutil==7.0.0'], check=True)
import transformers, safetensors, pybind11
print({'transformers': transformers.__version__, 'safetensors': safetensors.__version__,
       'pybind11': pybind11.__version__}, flush=True)

In [ ]:
ROOT = Path('/kaggle/temp/dee-source')
if ROOT.exists():
    assert str(ROOT.resolve()).startswith('/kaggle/temp/')
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--branch', 'opt/real-model-t1', '--single-branch',
                'https://github.com/so-nerdyy/dee.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', '--detach', '723c032d4f55f9f38bdaa02ef075e9088836bac8'], cwd=ROOT, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
assert commit == '723c032d4f55f9f38bdaa02ef075e9088836bac8', commit
DEE = ROOT / 'dee.cpp'
EVIDENCE = Path('/kaggle/working/ornith-milestone2-evidence')
EVIDENCE.mkdir(parents=True, exist_ok=True)
print({'repo': str(ROOT), 'commit': commit, 'dee': str(DEE)}, flush=True)

In [ ]:
candidates = []
for index_path in Path('/kaggle/input').rglob('model.safetensors.index.json'):
    config_path = index_path.parent / 'config.json'
    if config_path.is_file():
        config = json.loads(config_path.read_text())
        if config.get('model_type') == 'qwen3_5_moe':
            candidates.append(index_path.parent)
assert len(candidates) == 1, candidates
MODEL = candidates[0]
index = json.loads((MODEL / 'model.safetensors.index.json').read_text())
shards = sorted(set(index['weight_map'].values()))
assert len(shards) == 16, shards
missing = [name for name in shards if not (MODEL / name).is_file()]
assert not missing, missing
sizes = {name: (MODEL / name).stat().st_size for name in shards}
print(json.dumps({'model_dir': str(MODEL), 'tensor_count': len(index['weight_map']),
                  'shard_count': len(shards), 'shard_file_bytes': sizes,
                  'free_working_bytes': shutil.disk_usage('/kaggle/working').free}, indent=2), flush=True)

In [ ]:
BUILD = DEE / 'build-kaggle-cuda'
subprocess.run(['cmake', '-S', str(DEE), '-B', str(BUILD), '-G', 'Ninja',
                '-DDEE_CUDA=ON', '-DDEE_BUILD_TESTS=ON',
                '-DCMAKE_CUDA_ARCHITECTURES=75', '-DCMAKE_BUILD_TYPE=Release'], check=True)
subprocess.run(['cmake', '--build', str(BUILD), '--parallel', '4'], check=True)
subprocess.run(['ctest', '--test-dir', str(BUILD), '--output-on-failure'], check=True)
env = os.environ.copy(); env['DEE_BUILD_DIR'] = str(BUILD)
subprocess.run([sys.executable, str(DEE / 'pydee/setup.py'), 'build_ext', '--inplace', '--force'],
               cwd=DEE, env=env, check=True)
print('CUDA native build, native tests, and pydee binding passed', flush=True)

In [ ]:
layer0_report = EVIDENCE / 'ornith-layer0-regression.json'
subprocess.run([sys.executable, '-u', '-X', 'faulthandler', str(DEE / 'scripts/run_ornith_layer0_parity.py'),
                '--model-dir', str(MODEL), '--max-prompt-tokens', '4',
                '--report', str(layer0_report)], cwd=DEE, check=True)
layer0 = json.loads(layer0_report.read_text())
assert layer0['result'] == 'PASS', layer0
print({'layer0_result': layer0['result'], 'report': str(layer0_report)}, flush=True)

In [ ]:
def run_tee(command, log_path):
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=DEE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

command = [sys.executable, str(DEE / 'scripts/run_ornith_generation.py'),
           '--model-dir', str(MODEL), '--greedy', '--benchmark', '--reference-parity',
           '--require-dual-gpu', '--max-new-tokens', '4', '--cache-experts', '8',
           '--output-dir', str(EVIDENCE),
           '--prompt', 'Hello', '--prompt', '2+2=', '--prompt', 'Paris']
run_tee(command, EVIDENCE / 'full-run.log')
report = json.loads((EVIDENCE / 'ornith-milestone2-report.json').read_text())
assert report['result'] == 'PASS', report['result']
assert all(item['all_40_layers_executed'] for item in report['validation'])
assert all(item['generated_token_ids_exact'] for item in report['validation'])
assert all(len(item['candidate']['generated_token_ids']) >= 2 for item in report['validation'])
print(json.dumps({'result': report['result'],
                  'first_tokens': [item['candidate']['generated_token_ids'][0] for item in report['validation']],
                  'multi_token_ids': [item['candidate']['generated_token_ids'] for item in report['validation']],
                  'decoded': [item['candidate']['generated_text'] for item in report['validation']]},
                 ensure_ascii=False, indent=2), flush=True)

In [ ]:
manifest = {str(path.relative_to(EVIDENCE)): {'bytes': path.stat().st_size}
            for path in EVIDENCE.rglob('*') if path.is_file()}
(EVIDENCE / 'evidence-manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive('/kaggle/working/ornith-milestone2-evidence', 'gztar',
                              root_dir=EVIDENCE.parent, base_dir=EVIDENCE.name)
print(json.dumps({'final_status': 'PASS', 'evidence_dir': str(EVIDENCE),
                  'archive': archive, 'files': manifest}, indent=2), flush=True)